In [10]:
import pandas as pd
import re

In [11]:

# Dictionary to convert 3-letter amino acid codes to 1-letter codes

three_to_one = {
    'ALA': 'A', 'ARG': 'R', 'ASN': 'N', 'ASP': 'D', 'CYS': 'C',
    'GLN': 'Q', 'GLU': 'E', 'GLY': 'G', 'HIS': 'H', 'ILE': 'I',
    'LEU': 'L', 'LYS': 'K', 'MET': 'M', 'PHE': 'F', 'PRO': 'P',
    'SER': 'S', 'THR': 'T', 'TRP': 'W', 'TYR': 'Y', 'VAL': 'V',
    'GLX': 'Z', 'ASX': 'B', 'UNK': 'X'
}

# Function to extract amino acid and pKa from a log file
def extract_pka_data(log_text):
    pattern = re.compile(r'^\s*([A-Z]{3})\s+(\d+)\s+[A-Z]\s+([\d\.]+[*]?)')
    rows = []

    for line in log_text.splitlines():
        match = pattern.match(line)
        if match:
            aa3, pos, pka = match.groups()
            aa1 = three_to_one.get(aa3.upper(), '?')
            if '*' in pka:
                pka = pka.replace('*', '')
            rows.append({
                'amino acid': f'{aa1}{pos}',
                'pKa': float(pka),
                'pos': int(pos)  # for sorting
            })

    df = pd.DataFrame(rows)
    df = df.drop_duplicates(subset='amino acid', keep='first')
    df.sort_values('pos', inplace=True)
    df.drop(columns='pos', inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df

# Placeholder example
df = pd.DataFrame(columns=['amino acid', 'pKa'])




In [12]:
with open('./pdb2pqr_processed_pqrs/2r9w.log') as f:
    log_content = f.read()
    df = extract_pka_data(log_content)

df

,amino acid,pKa
0,D10,3.26
1,H13,6.48
2,R14,12.61
3,E21,4.59
4,K24,10.44
...,...,...
76,E333,4.45
77,K342,9.86
78,Y344,12.57
79,R349,10.91


In [13]:
df1 = df[df['pKa'] != 0]
df1

,amino acid,pKa
0,D10,3.26
1,H13,6.48
2,R14,12.61
3,E21,4.59
4,K24,10.44
...,...,...
76,E333,4.45
77,K342,9.86
78,Y344,12.57
79,R349,10.91


In [14]:
df1.to_csv('2r9w.csv')